In [ ]:
import torch
import numpy as np
import pickle
from openai import OpenAI
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.metrics import roc_curve, precision_recall_curve, auc
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
with open("data/190325_data.pkl", "rb") as f:
    train_chat_data = pickle.load(f)
    
# Remove service chat
del train_chat_data[-1000]

In [ ]:
with open("data/100925_data.pkl", "rb") as f:
    chat_data = pickle.load(f)
    
# Remove service chat
del chat_data[-1000]

In [ ]:
messages = []
labels = []
game_ids = []
user_ids = []
languages = []
for game_id, game_data in chat_data.items():
    if game_id in train_chat_data.keys():
        continue
    for message in game_data["messages"]:
        if message["userID"] == "GameMaster":
            if "won" in message["message"] or "surrendered" in message["message"] or "canceled" in message["message"] or "lost" in message["message"] or "timed out" in message["message"] or "disconnected" in message["message"]:
                break
            else:
                continue
        messages.append(message["message"])
        labels.append([int(not message["botID"]), message["botID"]])
        game_ids.append(message["gameID"])
        user_ids.append(message["userID"])
        if game_data["language"] is None:
            languages.append("en")
        else:
            languages.append(game_data["language"])
        
print(len(messages))
print(len(labels))

In [ ]:
class NaiveBayesClassifier:
    def __init__(self, path_to_state_dict):
        class MessageClassifier(torch.nn.Module):
            def __init__(self, input_size, hidden_sizes, output_size, dropout=0.75):
                super().__init__()
                self.activation = torch.nn.ReLU()
                layers = []
                current_size = input_size
                for hidden_size in hidden_sizes:
                    layers.append(torch.nn.Linear(current_size, hidden_size))
                    layers.append(torch.nn.BatchNorm1d(hidden_size))
                    layers.append(torch.nn.ReLU())
                    layers.append(torch.nn.Dropout(dropout))
                    current_size = hidden_size
                layers.append(torch.nn.Linear(current_size, output_size))
        
                self.model = torch.nn.Sequential(*layers)
        
            def forward(self, x):
                return self.model(x)

        
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.client = OpenAI(
                            api_key="EMPTY",
                            base_url="http://localhost:8000/v1"
                        )
        self.embedding_model = "intfloat/multilingual-e5-large-instruct"

        self.classifier = MessageClassifier(1024, [48, 24], 2, dropout=0.85)
        self.classifier.to(self.device)
        self.classifier.load_state_dict(torch.load(path_to_state_dict, weights_only=True))
        self.classifier.eval()

    def predict(self, message: str):
        # Classifies a message as human (negative score) or bot (positive score).
        embedding = torch.tensor(self.client.embeddings.create(input = [message], model=self.embedding_model).data[0].embedding)
        pred = self.classifier(embedding.unsqueeze(dim=0)).squeeze()
        return (pred[1]-pred[0]).item()
            

In [ ]:
classifier = NaiveBayesClassifier("models/run082.pth")

In [ ]:
classifier.predict("test")
classifier.predict("bot")
classifier.predict("hi")
classifier.predict("hey guys how are you")

In [ ]:
messages_with_i_dont_know= []
user_ids_with_i_dont_know = []
messages_with_idk = []
user_ids_with_idk = []
for message, user_id in zip(messages, user_ids):
    if "i dont know" in message:
        messages_with_i_dont_know.append(message)
        user_ids_with_i_dont_know.append(user_id)
    
    if "idk" in message:
        messages_with_idk.append(message)
        user_ids_with_idk.append(user_id)

In [ ]:
for user_id, message in zip(user_ids_with_idk[:3], messages_with_idk[:3]):
    print(message)
    print(classifier.predict(message))
    print("After changing")
    print(message.replace("idk", "i don't know"))
    print(classifier.predict(message.replace("idk", "i don't know")))

In [ ]:
messages_with_how_about_you= []
user_ids_with_how_about_you = []
messages_with_hbu = []
user_ids_with_hbu = []

message_with_bot = []
user_ids_with_bot = []
for message, user_id in zip(messages, user_ids):
    if "how about you" in message:
        messages_with_how_about_you.append(message)
        user_ids_with_how_about_you.append(user_id)
    
    if "hbu" in message:
        messages_with_hbu.append(message)
        user_ids_with_hbu.append(user_id)
    if "bot" in message:
        message_with_bot.append(message)
        user_ids_with_bot.append(user_id)

In [ ]:
print(classifier.predict("how about you"))
print(classifier.predict("hbu"))

In [ ]:
for user_id, message in zip(user_ids_with_how_about_you[:3], messages_with_how_about_you[:3]):
    print(message)
    print(classifier.predict(message))
    print("After changing")
    print(message.replace("how about you", "hbu"))
    print(classifier.predict(message.replace("how about you", "hbu")))

In [ ]:
for user_id, message in zip(user_ids_with_hbu[:4], messages_with_hbu[:4]):
    print(message)
    print(classifier.predict(message))
    print("After changing")
    print(message.replace("hbu", "how about you"))
    print(classifier.predict(message.replace("hbu", "how about you")))

In [ ]:
for user_id, message in zip(user_ids_with_bot[:10], message_with_bot[:10]):
    print(message)
    print(classifier.predict(message))
    #print("After changing")
    #print(message.replace("how about you", "hbu"))
    #print(classifier.predict(message.replace("how about you", "hbu")))

total = 0
for message in message_with_bot:
    total += classifier.predict(message)
print(f"Messages with 'bot' got an average score of {total/len(message_with_bot)}")

In [ ]:
random.seed(42)
i = 0
while i < 5:
    idx = random.randint(0, len(messages)-1)
    message = messages[idx]
    user_id = user_ids[idx]
    if user_id != '0':
        continue
    print(message)
    print(classifier.predict(message))
    print("After changing")
    message = message + " bot"
    print(message)
    print(classifier.predict(message))
    i += 1

In [ ]:
print(message_with_bot[9])
print(len(message_with_bot))

In [ ]:
random.seed(42)
n = 0
for idx in range(len(messages)-1, 0, -1):
    if user_ids[idx] == '0':
        if len(messages[idx].split(" ")) > 5:
            continue
        if messages[idx].split(" ")[0] == "hey" or messages[idx].split(" ")[0] == "hey,":
            continue
        if messages[idx][-1] != "?":
            continue
        print(messages[idx])
        print(classifier.predict(messages[idx]))
        print("After changing")
        print(f"hey {messages[idx]}")
        print(classifier.predict(f"hey {messages[idx]}"))
        n += 1
    if n == 4:
        break

In [ ]:
def get_player_scores(classifier, threshold=1):
    with open("data/190325_data.pkl", "rb") as f:
        chat_data = pickle.load(f)

    with open("data/100925_data.pkl", "rb") as f:
        test_data = pickle.load(f)

    evaluated_ids = []
    player_scores_per_game = []
    for game_id in tqdm(test_data.keys(), desc="Evaluating", total=len(test_data.keys())):
        if game_id in chat_data.keys():
            continue
        game = test_data[game_id]
        evaluated_ids.append(game_id)
        player_scores = defaultdict(float)
        
        for message in game["messages"]:
            if message["userID"] == "GameMaster":
                if "won" in message["message"] or "surrendered" in message["message"] or "canceled" in message["message"] or "lost" in message["message"] or "timed out" in message["message"] or "disconnected" in message["message"]:
                    break
                else:
                    continue
            else:
                if message["botID"]:
                    player_scores["bot"] += classifier.predict(message["message"])
                else:
                    player_scores[message["userID"]] += classifier.predict(message["message"])        

        if "bot" not in player_scores.keys():
            player_scores["bot"] = 0
        while len(player_scores.keys()) < 3:
            player_scores[f"simulated_player_{3-len(player_scores.keys())}"] = 0
        
        player_scores_per_game.append(player_scores)
    return player_scores_per_game, evaluated_ids

In [ ]:
def evaluate_on_old_games(player_scores_per_game, threshold=1):
    overall_TP = 0
    overall_FP = 0
    overall_false_game_ids = []
    overall_FN = 0

    seen_users_TP = 0
    seen_users_FP = 0
    seen_users_false_game_ids = []
    seen_users_FN = 0
    
    new_users_TP = 0
    new_users_FP = 0
    new_users_false_game_ids = []
    new_users_FN = 0


    with open("data/corresponding_game_ids.pkl", "rb") as f:
        game_ids = pickle.load(f)
    
    with open("data/users_in_train.pkl", "rb") as f:
        train_users = pickle.load(f)
        
    with open("data/users_in_val.pkl", "rb") as f:
        val_users = pickle.load(f)

    seen_users = train_users | val_users

    for game_scores, game_id in zip(player_scores_per_game, game_ids):

        colors = list(game_scores.keys())
        for player in colors:
            if player == "bot":
                continue

            bot_score = game_scores["bot"]
            human_score = game_scores[player]

            if bot_score - human_score > threshold:
                # Classifier would accuse the bot
                overall_TP += 1
                if player in seen_users:
                    seen_users_TP += 1
                else:
                    new_users_TP += 1
            elif human_score - bot_score > threshold:
                # Classifier would accuse a human
                overall_FP += 1
                if player in seen_users:
                    seen_users_FP += 1
                    seen_users_false_game_ids.append(game_id)
                else:
                    new_users_FP += 1
                    new_users_false_game_ids.append(game_id)
                overall_false_game_ids.append(game_id)
            else:
                # No accusation
                overall_FN += 1
                if player in seen_users:
                    seen_users_FN += 1
                    seen_users_false_game_ids.append(game_id)
                else:
                    new_users_FN += 1
                    new_users_false_game_ids.append(game_id)
                overall_false_game_ids.append(game_id)
    
    data = {
        "overall": {
            "TP": overall_TP,
            "FP": overall_FP,
            "FN": overall_FN,
            "ids": overall_false_game_ids
        },
        "seen": {
            "TP": seen_users_TP,
            "FP": seen_users_FP,
            "FN": seen_users_FN,
            "ids": seen_users_false_game_ids
        },
        "new": {
            "TP": new_users_TP,
            "FP": new_users_FP,
            "FN": new_users_FN,
            "ids": new_users_false_game_ids
        },
    }
    return data

In [ ]:
#player_scores_per_game, game_ids = get_player_scores(classifier)
#with open("data/precomputed_player_scores.pkl", "wb") as f:
#    pickle.dump(player_scores_per_game, f)
#with open("data/corresponding_game_ids.pkl", "wb") as f:
#    pickle.dump(game_ids, f)

with open("data/precomputed_player_scores.pkl", "rb") as f:
    player_scores_per_game = pickle.load(f)
with open("data/corresponding_game_ids.pkl", "rb") as f:
    game_ids = pickle.load(f)

In [ ]:
print(f"Evaluated {len(game_ids)} games")

In [ ]:
data = evaluate_on_old_games(player_scores_per_game, threshold=1)
overall_data = data["overall"]
tp = overall_data["TP"]
fp = overall_data["FP"]
fn = overall_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on all users regardless of if they have been seen or not using threshold 1")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
seen_data = data["seen"]
tp = seen_data["TP"]
fp = seen_data["FP"]
fn = seen_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on users in the train or validation set using threshold 1")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
new_data = data["new"]
tp = new_data["TP"]
fp = new_data["FP"]
fn = new_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on users not in the train or validation set using threshold 1")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
with open("data/testing_data_dict_threshold_1.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
data = evaluate_on_old_games(player_scores_per_game, threshold=3)
overall_data = data["overall"]
tp = overall_data["TP"]
fp = overall_data["FP"]
fn = overall_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on all users regardless of if they have been seen or not using threshold 3")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
seen_data = data["seen"]
tp = seen_data["TP"]
fp = seen_data["FP"]
fn = seen_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on users in the train or validation set using threshold 3")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
new_data = data["new"]
tp = new_data["TP"]
fp = new_data["FP"]
fn = new_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on users not in the train or validation set using threshold 3")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
with open("data/testing_data_dict_threshold_3.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
data = evaluate_on_old_games(player_scores_per_game, threshold=5)
overall_data = data["overall"]
tp = overall_data["TP"]
fp = overall_data["FP"]
fn = overall_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on all users regardless of if they have been seen or not using threshold 5")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
seen_data = data["seen"]
tp = seen_data["TP"]
fp = seen_data["FP"]
fn = seen_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on users in the train or validation set using threshold 5")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
new_data = data["new"]
tp = new_data["TP"]
fp = new_data["FP"]
fn = new_data["FN"]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
false_discovery_rate = fp / (tp + fp) if (tp + fp) > 0 else 0.0
miss_rate = fn / (tp + fn) if (tp + fn) > 0 else 0.0
print(f"Scores on users not in the train or validation set using threshold 5")
print(f"True Positives (TP):          {tp}")
print(f"False Positives (FP):         {fp}")
print(f"False Negatives (FN):         {fn}")
print()
print(f"Precision:                    {precision:.4f}")
print(f"Recall (TPR):                 {recall:.4f}")
print(f"F1 Score:                     {f1_score:.4f}")
print(f"False Discovery Rate (FDR):   {false_discovery_rate:.4f}")
print(f"Miss Rate (FNR):              {miss_rate:.4f}")

In [ ]:
with open("data/testing_data_dict_threshold_5.pkl", "wb") as f:
    pickle.dump(data, f)

In [ ]:
def evaluate_threshold_curve(player_scores_per_game):
    thresholds = np.linspace(0, 5, 100)
    precisions = []
    recalls = []
    f1_scores = []

    for threshold in thresholds:
        tp = 0
        fp = 0
        fn = 0

        for scores in player_scores_per_game:
            bot_score = scores["bot"]
            for player, human_score in scores.items():
                if player == "bot":
                    continue

                if bot_score - human_score > threshold:
                    tp += 1
                elif human_score - bot_score > threshold:
                    fp += 1
                else:
                    fn += 1

        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    plt.figure(figsize=(10, 6))
    plt.plot(thresholds, precisions, label="Precision", color='blue')
    plt.plot(thresholds, recalls, label="Recall", color='green')
    plt.plot(thresholds, f1_scores, label="F1 Score", color='red')
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.title("Classifier Performance vs Threshold")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return thresholds, precisions, recalls, f1_scores

In [ ]:
thresholds, precisions, recalls, f1_scores = evaluate_threshold_curve(player_scores_per_game)

In [ ]:
with open("data/val_games_player_scores.pkl", "wb") as f:
    pickle.dump(player_scores_per_game, f)

In [ ]:
def evaluate_thresholds_roc(player_scores_per_game, threshold_list):
    score_diffs = []
    labels = []

    for scores in player_scores_per_game:
        bot_score = scores["bot"]
        for player, human_score in scores.items():
            if player == "bot":
                continue
            score_diffs.append(bot_score - human_score)
            labels.append(1)

            score_diffs.append(human_score - bot_score)
            labels.append(0)

    score_diffs = np.array(score_diffs)
    labels = np.array(labels)

    fpr_base, tpr_base, _ = roc_curve(labels, score_diffs)
    roc_auc_base = auc(fpr_base, tpr_base)

    precision_base, recall_base, _ = precision_recall_curve(labels, score_diffs)
    pr_auc_base = auc(recall_base, precision_base)

    baseline_precision = np.mean(labels)

    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.plot(fpr_base, tpr_base, label=f"Base ROC AUC = {roc_auc_base:.4f}", color="black", linestyle="--")
    plt.plot([0, 1], [0, 1], linestyle=":", color="gray", label="Random Classifier (Baseline)")

    for thresh in threshold_list:
        preds = (score_diffs >= thresh).astype(int)
        tp = np.sum((preds == 1) & (labels == 1))
        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))
        tn = np.sum((preds == 0) & (labels == 0))

        fpr = fp / (fp + tn) if (fp + tn) else 0
        tpr = tp / (tp + fn) if (tp + fn) else 0
        plt.scatter(fpr, tpr, label=f"Threshold {thresh:.2f}")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True)



    plt.subplot(1, 2, 2)
    plt.plot(recall_base, precision_base, label=f"Base PR AUC = {pr_auc_base:.4f}", color="black", linestyle="--")
    plt.axhline(y=baseline_precision, color="gray", linestyle=":", label=f"Baseline Precision = {baseline_precision:.2f}")

    for thresh in threshold_list:
        preds = (score_diffs >= thresh).astype(int)
        tp = np.sum((preds == 1) & (labels == 1))
        fp = np.sum((preds == 1) & (labels == 0))
        fn = np.sum((preds == 0) & (labels == 1))

        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        plt.scatter(recall, precision, label=f"Threshold {thresh:.2f}")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.ylim(-0.05, 1.05)
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    return {
        "base_roc_auc": roc_auc_base,
        "base_pr_auc": pr_auc_base,
        "baseline_precision": baseline_precision
    }


In [ ]:
output = evaluate_thresholds_roc(player_scores_per_game, [0, 1, 2, 3, 5])

In [ ]:
with open("data/false_game_ids_threshold_3.pkl", "rb") as f:
    wrong_indices = pickle.load(f)
    
val_games = game_ids

In [ ]:
lengths = 0
long_wrong_games = []
for game in wrong_indices:
    game_messages = []
    game_data = chat_data[game]
    for message in game_data["messages"]:
        if message["userID"] == "GameMaster":
            if "won" in message["message"] or "surrendered" in message["message"] or "canceled" in message["message"] or "lost" in message["message"] or "timed out" in message["message"] or "disconnected" in message["message"]:
                break
            else:
                continue
        game_messages.append(message["message"])
    lengths += len(game_messages)
    if len(game_messages) >= 10:
        long_wrong_games.append(game)

print(lengths/len(wrong_indices))

In [ ]:
print(len(long_wrong_games))
print(len(wrong_indices))

In [ ]:
print(wrong_indices)

In [ ]:
english = 0
german = 0
for idx in long_wrong_games:
    if chat_data[idx]["language"] == "de":
        german += 1
    if chat_data[idx]["language"] == "en":
        english += 1
print(german)
print(english)

In [ ]:
with open("data/corresponding_game_ids.pkl", "rb") as f:
    tested_game_ids = pickle.load(f)

In [ ]:
from scipy.stats import pearsonr
def get_message_correlation():
    with open("data/190325_data.pkl", "rb") as f:
        chat_data = pickle.load(f)

    with open("data/100925_data.pkl", "rb") as f:
        test_data = pickle.load(f)

    mean_correlation = 0
    i = 0
    for game_id in tqdm(test_data.keys(), desc="Evaluating", total=len(test_data.keys())):
        game = test_data[game_id]

        players = set(message["userID"] for message in game["messages"])
        for player in players:
            scores = []
            for message in game["messages"]:
                if message["userID"] == "GameMaster":
                    if "won" in message["message"] or "surrendered" in message["message"] or "canceled" in message["message"] or "lost" in message["message"] or "timed out" in message["message"] or "disconnected" in message["message"]:
                        break
                    else:
                        continue
                if message["userID"] == player:
                    scores.append(classifier.predict(message["message"]))
            
            if len(scores) < 2:
                continue
            scores_next = scores[1:] + [0]
            mean_correlation += pearsonr(scores, scores_next)[0]
            i += 1
        
    return mean_correlation/i